In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install ipywidgets --upgrade

In [ ]:
import os
iskaggle = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '')

In [ ]:
creds = '{"username":"candicechristabel","key":"e82a68abd1ba3753b34794a8c3fa6274"}'

In [ ]:
from pathlib import Path

cred_path = Path('~/.kaggle/kaggle.json').expanduser()
if not cred_path.exists():
    cred_path.parent.mkdir(exist_ok=True)
    cred_path.write_text(creds)
    cred_path.chmod(0o600)

In [ ]:
path = Path('/kaggle/input/nlp-getting-started')
! pip install -q datasets

In [ ]:
df_train = pd.read_csv(path/'train.csv')
df_test = pd.read_csv(path/'test.csv')

In [ ]:
df_train

In [ ]:
df_test

In [ ]:
df_train.describe(include='object')

In [ ]:
train = df_train.drop(['id', 'target'], axis=1)
test = df_test.drop(['id'], axis=1)

In [ ]:
df_train['input'] = [str(row) for row in (train.to_dict(orient='records'))]
df_test['input'] = [str(row) for row in (test.to_dict(orient='records'))]
df_train.input.head()

In [ ]:
df_test.input.head()

In [ ]:
from datasets import Dataset,DatasetDict

ds_train = Dataset.from_pandas(df_train)
ds_test = Dataset.from_pandas(df_test)
ds_train

In [ ]:
ds_test

In [ ]:
model_nm = 'microsoft/deberta-v3-small'

from transformers import AutoModelForSequenceClassification,AutoTokenizer
tokz = AutoTokenizer.from_pretrained(model_nm)

In [ ]:
def tok_func(x): return tokz(x["input"])
tok_ds = ds_train.map(tok_func, batched=True)

row = tok_ds[0]
row['input'], row['input_ids']

In [ ]:
tok_ds = tok_ds.rename_columns({'target':'labels'})
print(tok_ds)

In [ ]:
df_test.describe()

In [ ]:
dds = tok_ds.train_test_split(0.25, seed=42)
dds

In [ ]:
print(df_test.head())

In [ ]:
from transformers import TrainingArguments,Trainer

bs = 128
epochs = 1
lr = 8e-5

In [ ]:
args = TrainingArguments('outputs', learning_rate=lr, warmup_ratio=0.1, lr_scheduler_type='cosine', fp16=True,
    eval_strategy="epoch", per_device_train_batch_size=bs, per_device_eval_batch_size=bs*2,
    num_train_epochs=epochs, weight_decay=0.01, report_to='none')

In [ ]:
# from sklearn.metrics import f1_score
# def my_metric(eval_pred): 
#     return {'F1': f1_score(*eval_pred, zero_division=0)}

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_nm, num_labels=2)
trainer = Trainer(model, args, train_dataset=dds['train'], eval_dataset=dds['test'],
                  tokenizer=tokz)

In [ ]:
trainer.train();

In [ ]:
tok_ds_test = ds_test.map(tok_func, batched=True)
preds = trainer.predict(tok_ds_test)
preds

In [ ]:
class_preds = np.argmax(preds.predictions, axis=-1)
sub = pd.read_csv('/kaggle/input/nlp-getting-started/sample_submission.csv')
sub['target'] = class_preds
sub

In [ ]:
sub.to_csv(f'submission.csv', index=False) 